# SO3.2-04 — Preflight do processamento populacional: beta 2000–2003

## Objetivo desta primeira execução

Iniciar a etapa operacional de construção dos produtos populacionais anuais
do SO3.2 para o período beta **2000–2003**.

Antes de transferir ou processar os grandes GeoTIFFs históricos do WorldPop,
esta primeira parte realiza um **preflight de transferência e acesso remoto**.

O objetivo é responder, com os próprios arquivos utilizados pelo projeto:

- qual o volume real dos arquivos de idade/sexo para 2000–2003;
- qual o volume da população total;
- se o servidor WorldPop oferece acesso por intervalos HTTP (`Range`);
- se os GeoTIFFs podem ser abertos remotamente pelo GDAL/rasterio;
- quais são as dimensões, blocagem, compressão e tipo de dado de um arquivo
  representativo;
- se o espaço temporário disponível no ambiente é compatível com estratégias
  de processamento local.

**Nenhum GeoTIFF completo é baixado nesta etapa.**

A decisão entre processamento no Colab, servidor local ou arquitetura híbrida
será tomada a partir dos resultados deste preflight.

## 1. Ambiente e caminhos

O inventário produzido no SO3.2-03 é reutilizado diretamente a partir do
diretório de logs no Google Drive.

In [1]:
from google.colab import drive
from pathlib import Path
import shutil
import time

import pandas as pd
import requests
import rasterio
from rasterio.windows import Window

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/Cemaden")
PROJECT_ROOT = DRIVE_ROOT / "PRAIS4_SO3_BR"
SO32_ROOT = PROJECT_ROOT / "SO3.2"
SO32_LOGS = SO32_ROOT / "logs"

SEX_INVENTORY = SO32_LOGS / "so32_03_worldpop_sex_archive_inventory.csv"

assert SEX_INVENTORY.exists(), (
    f"Inventário do SO3.2-03 não encontrado: {SEX_INVENTORY}"
)

print(f"Inventário: {SEX_INVENTORY}")
print(f"Logs      : {SO32_LOGS}")

Mounted at /content/drive
Inventário: /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_03_worldpop_sex_archive_inventory.csv
Logs      : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs


## 2. Seleção do período beta

O período 2000–2003 é o primeiro quadriênio do PRAIS e será utilizado como
bloco beta para validar posteriormente todo o fluxo anual e quadrianual.

In [2]:
BETA_YEARS = [2000, 2001, 2002, 2003]

sex_inventory = pd.read_csv(SEX_INVENTORY)

beta_sex = (
    sex_inventory[sex_inventory["year"].isin(BETA_YEARS)]
    .copy()
    .sort_values(["year", "sex", "age_class"])
    .reset_index(drop=True)
)

print(f"Arquivos idade/sexo no beta: {len(beta_sex)}")

assert len(beta_sex) == 4 * 2 * 18, (
    "Número inesperado de arquivos idade/sexo para 2000–2003."
)

beta_sex.groupby(["year", "sex"]).size()

Arquivos idade/sexo no beta: 144


year  sex
2000  f      18
      m      18
2001  f      18
      m      18
2002  f      18
      m      18
2003  f      18
      m      18
dtype: int64

## 3. Consulta do tamanho remoto dos arquivos

A função abaixo utiliza inicialmente `HEAD`. Caso o servidor não informe o
tamanho dessa forma, é feita uma requisição mínima com `Range: bytes=0-0`.

O conteúdo dos GeoTIFFs não é transferido integralmente.

In [3]:
session = requests.Session()
session.headers.update({
    "User-Agent": "PRAIS4-SO3.2-Brazil/1.0 (technical preflight; no bulk raster download)"
})

def remote_file_info(url, timeout=60):
    record = {
        "url": url,
        "http_status": None,
        "content_length_bytes": None,
        "accept_ranges": None,
        "etag": None,
        "last_modified": None,
        "method": None,
    }

    try:
        response = session.head(url, allow_redirects=True, timeout=timeout)
        record["http_status"] = response.status_code
        record["accept_ranges"] = response.headers.get("Accept-Ranges")
        record["etag"] = response.headers.get("ETag")
        record["last_modified"] = response.headers.get("Last-Modified")
        record["method"] = "HEAD"
        length = response.headers.get("Content-Length")
        if length is not None:
            record["content_length_bytes"] = int(length)
    except requests.RequestException:
        pass

    if record["content_length_bytes"] is None:
        try:
            response = session.get(
                url,
                headers={"Range": "bytes=0-0"},
                stream=True,
                allow_redirects=True,
                timeout=timeout,
            )
            record["http_status"] = response.status_code
            record["accept_ranges"] = response.headers.get("Accept-Ranges") or record["accept_ranges"]
            record["etag"] = response.headers.get("ETag") or record["etag"]
            record["last_modified"] = response.headers.get("Last-Modified") or record["last_modified"]
            record["method"] = "RANGE_GET"

            content_range = response.headers.get("Content-Range")
            if content_range and "/" in content_range:
                total = content_range.split("/")[-1]
                if total.isdigit():
                    record["content_length_bytes"] = int(total)

            response.close()
        except requests.RequestException:
            pass

    return record

### 3.1 Arquivos de idade/sexo

São consultados os 144 arquivos correspondentes aos quatro anos do beta
(18 classes etárias × 2 sexos × 4 anos).

In [4]:
remote_records = []

for i, row in beta_sex.iterrows():
    info = remote_file_info(row["url"])
    remote_records.append({
        "year": int(row["year"]),
        "sex": row["sex"],
        "age_class": int(row["age_class"]),
        "filename": row["filename"],
        **info,
    })

    if (i + 1) % 18 == 0:
        print(f"Consultados {i + 1:3d} de {len(beta_sex)} arquivos")

beta_remote = pd.DataFrame(remote_records)
beta_remote["size_gb"] = beta_remote["content_length_bytes"] / (1024 ** 3)

beta_remote.head()

Consultados  18 de 144 arquivos
Consultados  36 de 144 arquivos
Consultados  54 de 144 arquivos
Consultados  72 de 144 arquivos
Consultados  90 de 144 arquivos
Consultados 108 de 144 arquivos
Consultados 126 de 144 arquivos
Consultados 144 de 144 arquivos


,year,sex,age_class,filename,url,http_status,content_length_bytes,accept_ranges,etag,last_modified,method,size_gb
0,2000,f,0,bra_f_0_2000.tif,https://data.worldpop.org/GIS/AgeSex_structure...,200,3783408809,bytes,"""e1823ca9-5781b9dc8ab77""","Sat, 13 Oct 2018 12:48:10 GMT",HEAD,3.523574
1,2000,f,1,bra_f_1_2000.tif,https://data.worldpop.org/GIS/AgeSex_structure...,200,3783347263,bytes,"""e1814c3f-5781b7efa00f0""","Sat, 13 Oct 2018 12:39:33 GMT",HEAD,3.523517
2,2000,f,5,bra_f_5_2000.tif,https://data.worldpop.org/GIS/AgeSex_structure...,200,3783292288,bytes,"""e1807580-5781b6cad87d1""","Sat, 13 Oct 2018 12:34:26 GMT",HEAD,3.523466
3,2000,f,10,bra_f_10_2000.tif,https://data.worldpop.org/GIS/AgeSex_structure...,200,3783253076,bytes,"""e17fdc54-5781b3b7d7ad4""","Sat, 13 Oct 2018 12:20:41 GMT",HEAD,3.523429
4,2000,f,15,bra_f_15_2000.tif,https://data.worldpop.org/GIS/AgeSex_structure...,200,3783253271,bytes,"""e17fdd17-5781b667f7de3""","Sat, 13 Oct 2018 12:32:42 GMT",HEAD,3.523429


### 3.2 População total

A população total é uma entrada independente das populações feminina e
masculina. Aqui são consultados os quatro GeoTIFFs anuais do produto
WorldPop Global1 correspondente.

In [5]:
TOTAL_BASE = "https://data.worldpop.org/GIS/Population/Global_2000_2020"

total_records = []

for year in BETA_YEARS:
    url = f"{TOTAL_BASE}/{year}/BRA/bra_ppp_{year}.tif"
    info = remote_file_info(url)
    total_records.append({
        "year": year,
        "filename": f"bra_ppp_{year}.tif",
        **info,
    })

beta_total = pd.DataFrame(total_records)
beta_total["size_gb"] = beta_total["content_length_bytes"] / (1024 ** 3)

beta_total

,year,filename,url,http_status,content_length_bytes,accept_ranges,etag,last_modified,method,size_gb
0,2000,bra_ppp_2000.tif,https://data.worldpop.org/GIS/Population/Globa...,200,4353412605,bytes,"""1037bcdfd-57b04eec9b433""","Mon, 19 Nov 2018 13:59:59 GMT",HEAD,4.054431
1,2001,bra_ppp_2001.tif,https://data.worldpop.org/GIS/Population/Globa...,200,4366171230,bytes,"""1043e7c5e-57b0615bd71ea""","Mon, 19 Nov 2018 15:22:28 GMT",HEAD,4.066314
2,2002,bra_ppp_2002.tif,https://data.worldpop.org/GIS/Population/Globa...,200,4366081662,bytes,"""1043d1e7e-57b06c7664147""","Mon, 19 Nov 2018 16:12:08 GMT",HEAD,4.066230
3,2003,bra_ppp_2003.tif,https://data.worldpop.org/GIS/Population/Globa...,200,4369498005,bytes,"""104713f95-57b0770283568""","Mon, 19 Nov 2018 16:59:20 GMT",HEAD,4.069412


## 4. Volume de transferência estimado

Esta síntese quantifica o volume necessário caso os arquivos sejam
transferidos integralmente.

Ela não implica que a estratégia final será o download integral.

In [6]:
sex_volume = (
    beta_remote
    .groupby(["year", "sex"], as_index=False)
    .agg(
        files=("filename", "count"),
        volume_gb=("size_gb", "sum"),
        mean_file_gb=("size_gb", "mean"),
    )
)

year_volume = (
    beta_remote
    .groupby("year", as_index=False)
    .agg(
        sex_files=("filename", "count"),
        age_sex_volume_gb=("size_gb", "sum"),
    )
    .merge(
        beta_total[["year", "size_gb"]]
        .rename(columns={"size_gb": "total_population_gb"}),
        on="year",
        how="left",
    )
)

year_volume["combined_volume_gb"] = (
    year_volume["age_sex_volume_gb"]
    + year_volume["total_population_gb"]
)

print("Volume por sexo/ano:")
display(sex_volume)

print("\nVolume total por ano:")
display(year_volume)

print(
    "\nVolume estimado do beta 2000–2003:",
    f"{year_volume['combined_volume_gb'].sum():.2f} GB"
)

Volume por sexo/ano:


,year,sex,files,volume_gb,mean_file_gb
0,2000,f,18,63.423091,3.523505
1,2000,m,18,63.423105,3.523506
2,2001,f,18,69.798279,3.877682
3,2001,m,18,69.797713,3.877651
4,2002,f,18,64.300047,3.572225
5,2002,m,18,64.300215,3.572234
6,2003,f,18,64.828004,3.601556
7,2003,m,18,64.828030,3.601557



Volume total por ano:


,year,sex_files,age_sex_volume_gb,total_population_gb,combined_volume_gb
0,2000,36,126.846197,4.054431,130.900628
1,2001,36,139.595992,4.066314,143.662306
2,2002,36,128.600263,4.066230,132.666493
3,2003,36,129.656034,4.069412,133.725446



Volume estimado do beta 2000–2003: 540.95 GB


## 5. Suporte a HTTP Range

O acesso por intervalos é importante para avaliar se o GDAL pode consultar
partes do GeoTIFF remoto sem transferir o arquivo inteiro.

A existência de `Range` **não garante**, por si só, que o processamento
integral remoto seja eficiente; ela apenas permite testar essa possibilidade.

In [7]:
range_summary = pd.Series({
    "Arquivos idade/sexo consultados": len(beta_remote),
    "HTTP 200/206": beta_remote["http_status"].isin([200, 206]).sum(),
    "Tamanho identificado": beta_remote["content_length_bytes"].notna().sum(),
    "Accept-Ranges = bytes": (
        beta_remote["accept_ranges"]
        .fillna("")
        .str.lower()
        .eq("bytes")
        .sum()
    ),
})

range_summary

,0
Arquivos idade/sexo consultados,144
HTTP 200/206,144
Tamanho identificado,144
Accept-Ranges = bytes,144


## 6. Capacidade temporária do ambiente

O espaço disponível no runtime atual é comparado ao volume de um conjunto
sexo/ano e de um ano completo.

In [8]:
disk = shutil.disk_usage("/content")

disk_summary = pd.Series({
    "Espaço total /content (GB)": disk.total / (1024 ** 3),
    "Espaço usado /content (GB)": disk.used / (1024 ** 3),
    "Espaço livre /content (GB)": disk.free / (1024 ** 3),
    "Maior volume sexo/ano (GB)": sex_volume["volume_gb"].max(),
    "Maior volume idade/sexo por ano (GB)": year_volume["age_sex_volume_gb"].max(),
})

disk_summary

,0
Espaço total /content (GB),107.715084
Espaço usado /content (GB),20.318985
Espaço livre /content (GB),87.380474
Maior volume sexo/ano (GB),69.798279
Maior volume idade/sexo por ano (GB),139.595992


## 7. Teste de abertura remota de um GeoTIFF

Se o servidor aceitar acesso por intervalos, um arquivo representativo de
2000 é aberto por `/vsicurl/`.

São consultados apenas metadados e uma pequena janela central. O objetivo é
avaliar se a estrutura do TIFF é compatível com leitura remota por blocos.

In [11]:
sample = (
    beta_remote[
        (beta_remote["year"] == 2000)
        & (beta_remote["sex"] == "f")
    ]
    .sort_values("age_class")
    .iloc[0]
)

sample_url = sample["url"]

http_range_announced = (
    str(sample["accept_ranges"]).lower() == "bytes"
)

print(f"Arquivo de teste       : {sample['filename']}")
print(f"Tamanho                : {sample['size_gb']:.3f} GB")
print(f"HTTP Range anunciado   : {sample['accept_ranges']}")

remote_read_supported = False
remote_read_error = None

if http_range_announced:

    vsi_url = f"/vsicurl/{sample_url}"

    env_options = {
        "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
        "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    }

    try:

        t0 = time.perf_counter()

        with rasterio.Env(**env_options):

            with rasterio.open(vsi_url) as src:

                metadata_elapsed = (
                    time.perf_counter() - t0
                )

                raster_info = {
                    "driver": src.driver,
                    "width": src.width,
                    "height": src.height,
                    "count": src.count,
                    "dtype": src.dtypes[0],
                    "crs": (
                        src.crs.to_string()
                        if src.crs
                        else None
                    ),
                    "transform": str(src.transform),
                    "nodata": src.nodata,
                    "tiled": src.is_tiled,
                    "block_shapes": str(src.block_shapes),
                    "compression": (
                        src.compression.name
                        if src.compression is not None
                        else None
                    ),
                }

                win_w = min(512, src.width)
                win_h = min(512, src.height)

                col_off = max(
                    0,
                    (src.width - win_w) // 2
                )

                row_off = max(
                    0,
                    (src.height - win_h) // 2
                )

                window = Window(
                    col_off,
                    row_off,
                    win_w,
                    win_h
                )

                t1 = time.perf_counter()

                sample_data = src.read(
                    1,
                    window=window,
                    masked=True
                )

                window_elapsed = (
                    time.perf_counter() - t1
                )

        remote_read_supported = True

        print("\nMetadados:")
        display(pd.Series(raster_info))

        print(
            f"\nTempo para abrir metadados: "
            f"{metadata_elapsed:.2f} s"
        )

        print(
            f"Tempo para ler janela "
            f"{win_w}×{win_h}: "
            f"{window_elapsed:.2f} s"
        )

        print(
            "Pixels válidos na janela:",
            int(sample_data.count())
        )

    except Exception as exc:

        remote_read_error = str(exc)

        print("\nLeitura remota GDAL: NÃO SUPORTADA")
        print(f"Motivo: {remote_read_error}")

else:

    print(
        "\nLeitura remota GDAL não testada: "
        "o servidor não anunciou suporte a HTTP Range."
    )

Arquivo de teste       : bra_f_0_2000.tif
Tamanho                : 3.524 GB
HTTP Range anunciado   : bytes

Leitura remota GDAL: NÃO SUPORTADA
Motivo: Range downloading not supported by this server!


## 8. Registro do preflight

Os resultados são registrados no diretório de logs. Esses arquivos são
pequenos e podem posteriormente ser copiados para `so3.2/metadata/` no
repositório.

In [10]:
remote_output = SO32_LOGS / "so32_04_beta_remote_file_inventory.csv"
volume_output = SO32_LOGS / "so32_04_beta_transfer_summary.csv"
total_output = SO32_LOGS / "so32_04_beta_total_population_files.csv"

beta_remote.to_csv(remote_output, index=False)
year_volume.to_csv(volume_output, index=False)
beta_total.to_csv(total_output, index=False)

print(f"Inventário remoto : {remote_output}")
print(f"Resumo de volume  : {volume_output}")
print(f"População total   : {total_output}")

Inventário remoto : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_04_beta_remote_file_inventory.csv
Resumo de volume  : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_04_beta_transfer_summary.csv
População total   : /content/drive/MyDrive/Cemaden/PRAIS4_SO3_BR/SO3.2/logs/so32_04_beta_total_population_files.csv


## 9. Próxima decisão

Esta etapa não escolhe automaticamente o ambiente de processamento.

A decisão será tomada a partir de:

1. volume real de transferência do beta;
2. espaço temporário disponível;
3. suporte a HTTP Range;
4. estrutura interna dos GeoTIFFs;
5. desempenho da leitura remota de uma pequena janela.

A partir desses resultados será definida a implementação do motor que irá
construir e preservar, para cada ano:

- `population_total`;
- `population_female`;
- `population_male`.

O primeiro processamento integral permanecerá o quadriênio **2000–2003**.

## Síntese

O processamento integral dos arquivos WorldPop de idade/sexo por download
simultâneo não é adequado ao ambiente Colab utilizado, devido ao volume de
dados envolvido.

Embora o servidor remoto anuncie suporte a HTTP Range, a leitura por blocos
via GDAL `/vsicurl/` não foi suportada na execução realizada.

Dessa forma, a construção das populações feminina e masculina será realizada
por processamento sequencial dos arquivos de origem, com armazenamento
temporário e descarte após incorporação ao acumulador anual.

Os produtos populacionais anuais consolidados serão preservados para
reutilização nas etapas posteriores do SO3.2.